# Vi-PPE V5 Judge Rerun

V5 fixes prompt JSON stability, criteria tie behavior, output token budget, smoke metrics limits, prompt hashing, and invalid swap aggregation.

## 1. Setup

In [ ]:
from pathlib import Path
import json

ROOT = Path('/content/Vi-PPE-mini')
if ROOT.exists():
    %cd /content/Vi-PPE-mini
else:
    print('Repo path /content/Vi-PPE-mini not found. Clone or mount it before continuing.')

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

In [ ]:
!python - <<'PY'
import torch
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
PY
!git rev-parse --short HEAD || true

## 2. Dataset And Config Check

In [ ]:
for path in [
    'data/processed/pairs_test_llm_v4.jsonl',
    'data/processed/bias_subset_llm_v4.jsonl',
]:
    p = Path(path)
    print(path, sum(1 for _ in p.open(encoding='utf-8')))

!python - <<'PY'
import yaml
cfg = yaml.safe_load(open('configs/models.yaml', encoding='utf-8'))
print(cfg['models']['qwen25_3b_a100_40gb'])
PY

## 3. Smoke Run With Matching Metrics Limit

In [ ]:
!python scripts/05_run_judge.py \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --model qwen25_3b_a100_40gb \
  --template auto_criteria_by_domain \
  --run-id qwen25_criteria_test_llm_v5_a100_b16_smoke \
  --limit 8 \
  --resume

In [ ]:
!python scripts/06_compute_metrics.py \
  --judgments results/judgments/qwen25_criteria_test_llm_v5_a100_b16_smoke.jsonl \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --run-id qwen25_criteria_test_llm_v5_a100_b16_smoke \
  --limit 8

## 4. Full Core Runs

In [ ]:
!python scripts/05_run_judge.py \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --model qwen25_3b_a100_40gb \
  --template baseline_generic_vi \
  --run-id qwen25_baseline_test_llm_v5_a100_b16 \
  --resume

In [ ]:
!python scripts/05_run_judge.py \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --model qwen25_3b_a100_40gb \
  --template auto_criteria_by_domain \
  --run-id qwen25_criteria_test_llm_v5_a100_b16 \
  --resume

In [ ]:
!python scripts/06_compute_metrics.py \
  --judgments results/judgments/qwen25_baseline_test_llm_v5_a100_b16.jsonl \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --run-id qwen25_baseline_test_llm_v5_a100_b16

!python scripts/06_compute_metrics.py \
  --judgments results/judgments/qwen25_criteria_test_llm_v5_a100_b16.jsonl \
  --dataset data/processed/pairs_test_llm_v4.jsonl \
  --run-id qwen25_criteria_test_llm_v5_a100_b16

## 5. Bias Runs

In [ ]:
!python scripts/05_run_judge.py \
  --dataset data/processed/bias_subset_llm_v4.jsonl \
  --model qwen25_3b_a100_40gb \
  --template baseline_generic_vi \
  --run-id qwen25_bias_baseline_llm_v5_a100_b16 \
  --resume

!python scripts/05_run_judge.py \
  --dataset data/processed/bias_subset_llm_v4.jsonl \
  --model qwen25_3b_a100_40gb \
  --template criteria_bias_mitigated_vi \
  --run-id qwen25_bias_mitigated_llm_v5_a100_b16 \
  --resume

In [ ]:
!python scripts/06_compute_metrics.py \
  --judgments results/judgments/qwen25_bias_baseline_llm_v5_a100_b16.jsonl \
  --dataset data/processed/bias_subset_llm_v4.jsonl \
  --run-id qwen25_bias_baseline_llm_v5_a100_b16 \
  --bias

!python scripts/06_compute_metrics.py \
  --judgments results/judgments/qwen25_bias_mitigated_llm_v5_a100_b16.jsonl \
  --dataset data/processed/bias_subset_llm_v4.jsonl \
  --run-id qwen25_bias_mitigated_llm_v5_a100_b16 \
  --bias

## 6. Summary And Backup

In [ ]:
run_ids = [
    'qwen25_baseline_test_llm_v5_a100_b16',
    'qwen25_criteria_test_llm_v5_a100_b16',
    'qwen25_bias_baseline_llm_v5_a100_b16',
    'qwen25_bias_mitigated_llm_v5_a100_b16',
]

for run_id in run_ids:
    path = Path(f'results/metrics/{run_id}_summary.json')
    data = json.loads(path.read_text(encoding='utf-8'))
    print('\n===', run_id, '===')
    for key in ['num_pairs', 'num_judgments', 'parse_success_rate', 'pairwise_accuracy', 'macro_accuracy', 'lower_bound_domain_score', 'swap_consistency', 'invalid_count', 'inconsistent_count']:
        print(key + ':', data.get(key))
    print('domain_accuracy:', data.get('domain_accuracy'))

In [ ]:
!mkdir -p /content/drive/MyDrive/Vi-PPE-mini/results_llm_v5_a100_b16 /content/drive/MyDrive/Vi-PPE-mini/checkpoints_llm_v5_a100_b16
!cp -r results/judgments /content/drive/MyDrive/Vi-PPE-mini/results_llm_v5_a100_b16/
!cp -r results/metrics /content/drive/MyDrive/Vi-PPE-mini/results_llm_v5_a100_b16/
!cp -r results/figures /content/drive/MyDrive/Vi-PPE-mini/results_llm_v5_a100_b16/ || true
!tar -czf /content/drive/MyDrive/Vi-PPE-mini/checkpoints_llm_v5_a100_b16/after_v5_$(date +%Y%m%d_%H%M%S).tar.gz results configs data/processed reports prompts src scripts